# Diviser pour régner — Exercice 3 : l'exponentiation rapide

**Notions travaillées :** compter les opérations d'un algorithme, récursivité, éviter de **recalculer** deux fois la même chose.

On compte les multiplications grâce à une variable globale `nb_mult`, augmentée de 1 à chaque multiplication.

## Question 1 : la force brute

```python
def power_bf(x, n):
    p = 1
    for i in range(n):
        p = p*x
    return p
```

La boucle tourne $n$ fois et fait **une** multiplication à chaque tour : **$n$ multiplications**. (On pourrait en économiser une en partant de `p = x`, ce qui fait $n - 1$.)
Le coût est donc **linéaire** : doubler $n$ double le travail.

In [1]:
nb_mult = 0

def mult(a, b):
    global nb_mult
    nb_mult += 1
    return a * b


def power_bf(x, n):
    p = 1
    for i in range(n):
        p = mult(p, x)
    return p


nb_mult = 0
print(power_bf(2, 100), "-> multiplications :", nb_mult)

1267650600228229401496703205376 -> multiplications : 100


## Question 2 : la version « maladroite »

On rappelle que
$x^n = \begin{cases} x & \text{si } n = 1 \\ x^{n/2}\cdot x^{n/2} & \text{si } n \text{ est pair} \\ x \cdot x^{n-1} & \text{si } n \text{ est impair} \end{cases}$

### (a) Pourquoi la ligne 5 est-elle une mauvaise idée ?

```python
return power_dac(x, n//2)*power_dac(x, n//2)      # ligne 5
```

Le résultat est **juste**, mais on calcule **deux fois exactement la même chose** : $x^{n/2}$. Et chacun de ces deux appels refait lui-même deux appels identiques, et ainsi de suite.
Le nombre d'appels **double à chaque niveau** : avec $\log(n)$ niveaux, on obtient environ $2^{\log(n)} = n$ appels. On revient donc à un coût **linéaire**, comme la force brute : diviser le problème n'a servi à rien.

On le vérifie en comptant :

In [2]:
def power_dac(x, n):
    if n == 1:
        return x
    if n % 2 == 0:
        return mult(power_dac(x, n//2), power_dac(x, n//2))
    else:
        return mult(x, power_dac(x, n-1))


nb_mult = 0
power_dac(2, 100)
print("Version maladroite, n = 100 :", nb_mult, "multiplications")

Version maladroite, n = 100 : 99 multiplications


### (b) Correction

On calcule $x^{n/2}$ **une seule fois**, on le **stocke dans une variable**, puis on l'élève au carré :

In [3]:
def power_dac(x, n):
    if n == 1:
        return x
    if n % 2 == 0:
        y = power_dac(x, n//2)        # un seul appel récursif
        return mult(y, y)
    else:
        return mult(x, power_dac(x, n-1))


nb_mult = 0
print(power_dac(2, 100) == 2**100, "-> multiplications :", nb_mult)

True -> multiplications : 8


### (c) Coût pour $n = 100$

On suit les appels à la main. Chaque flèche correspond à **une** multiplication :

| $n$ | parité | appel suivant | multiplication |
|:---|:---|:---|:---|
| 100 | pair | $x^{50}$ | $y \cdot y$ |
| 50 | pair | $x^{25}$ | $y \cdot y$ |
| 25 | impair | $x^{24}$ | $x \cdot (\ldots)$ |
| 24 | pair | $x^{12}$ | $y \cdot y$ |
| 12 | pair | $x^{6}$ | $y \cdot y$ |
| 6 | pair | $x^{3}$ | $y \cdot y$ |
| 3 | impair | $x^{2}$ | $x \cdot (\ldots)$ |
| 2 | pair | $x^{1}$ | $y \cdot y$ |
| 1 | — | cas de base | aucune |

**Total : 8 multiplications**, au lieu de 100 pour la force brute (et 99 pour la version maladroite).

💡 **En général :** chaque étape « pair » divise $n$ par 2, et une étape « impair » est toujours suivie d'une étape « pair ». Il y a donc au plus $2\log(n)$ multiplications : le coût est **logarithmique**.

In [4]:
print(f"{'n':>8} | {'force brute':>11} | {'maladroite':>10} | {'rapide':>6}")


def power_maladroite(x, n):
    if n == 1:
        return x
    if n % 2 == 0:
        return mult(power_maladroite(x, n//2), power_maladroite(x, n//2))
    return mult(x, power_maladroite(x, n-1))


for n in [10, 100, 1000, 10000]:
    res = []
    for f in [power_bf, power_maladroite, power_dac]:
        nb_mult = 0
        f(1, n)
        res.append(nb_mult)
    print(f"{n:>8} | {res[0]:>11} | {res[1]:>10} | {res[2]:>6}")

       n | force brute | maladroite | rapide
      10 |          10 |          9 |      4
     100 |         100 |         99 |      8
    1000 |        1000 |        999 |     14
   10000 |       10000 |       9999 |     17


💡 **À retenir :** dans un algorithme « diviser pour régner », il faut que les sous-problèmes soient **différents**. Si on résout deux fois le même sous-problème, on perd tout le bénéfice de la division.